<a href="https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/notebooks/w03_data_contract%20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one pseudonymized content item's search performance on one calendar day, for one pseudonymized client.**
Grain: `(report_date, client_id, content_id)` — the documented grain of `fact_content_daily_performance` (confirmed against `skills/flyrank/flyrank-data/SKILL.md`, verified with a query below).

**Time window:** a single mid-panel month, `month=2026-03`. The panel runs `2025-01-27` -> `2026-06-30` (~17 months); `2026-06` is the sealed `_sample`/final month and is never used to build label logic here — the last month is the natural outcome window of any past->future proxy.


In [7]:
%pip install -q duckdb

import duckdb
con = duckdb.connect()

# Never hardcode the token -- pull it from Colab Secrets (key icon, left panel). This repo is public.
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# Schema discovery FIRST. skills/flyrank/flyrank-data/SKILL.md confirms the position column is
# gsc_avg_position -- everything else below (gsc_clicks, gsc_impressions, gsc_ctr,
# ga4_data_available) follows the same documented naming pattern but isn't spelled out verbatim
# in the skill file, so this DESCRIBE is the actual source of truth -- check it before trusting
# any query further down, and fix names here if they differ.
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MID_MONTH}')").show()


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [8]:
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MID_MONTH}')").df()[['column_name','column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**The 5 contract answers:**

1. **What one row means:** one content item's observed GSC (+ GA4, where available) performance on one day, for one client (Section 1).
2. **Table(s) used:** `fact_content_daily_performance` only, `month=2026-03` partition. `dim_content`/`dim_clients` may be joined read-only for context, never for the grain.
3. **Time window:** `month=2026-03` (mid-panel). `month=2026-06` (the `_sample`) stays sealed.
4. **Predict / rank (label or proxy):** rank content items within the month by *directional risk of organic visibility decline* — a proxy built from the drop in observed GSC impressions between the first half of the month (days 1-15) and the second half (days 16-end), computed entirely from within-month rows. This is a proxy, not ground truth — directional, decision-support framing only.
5. **Deliberately excluded:** `fact_content_query_90d`. Per `skills/flyrank/flyrank-data/SKILL.md`, it's a different grain entirely — a fixed 90-day rolling window (not a monthly partition), its per-content context columns repeat on every row (need `ANY_VALUE()`, never `SUM()`), and its 90-day window overlaps the snapshot's final months, so only `*_prev30`-style columns would even be safe there. Folding it into this month-grain contract would break the "one row = one thing" rule. GA4-only columns are also excluded from the feature set — this lane is GSC-specific.

**Field buckets:**
- **Feature:** `gsc_impressions`, `gsc_clicks`, `gsc_ctr`, `gsc_avg_position` (confirm exact spelling via the `DESCRIBE` above), restricted to the first-half window only
- **Label / proxy:** second-half vs. first-half impression change — computed in Section 3, not a warehouse column
- **Context:** `client_id`, `content_id`, `report_date` — for grouping/filtering only, never fed to a model
- **Excluded:** `fact_content_query_90d` (grain mismatch), GA4-only metrics (out of lane), `month=2026-06` (sealed test month)


In [9]:
# No new query needed here -- Section 2 is the classification itself.
# The DESCRIBE output from Section 1 is what these bucket assignments are checked against.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query A — grain check
Claim: one row really is one `(report_date, client_id, content_id)`.

In [10]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM read_parquet('{MID_MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""")
dupes.show()
# Expect zero rows back. If rows come back, the grain claim in Section 1 is wrong -- fix the
# contract, don't ignore the query (writing-data-contracts skill, "Verify every claim").


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────┐
│ report_date │ client_hash_id │ content_hash_id │ n_rows │
│    date     │    varchar     │     varchar     │ int64  │
├─────────────┴────────────────┴─────────────────┴────────┤
│                         0 rows                          │
└─────────────────────────────────────────────────────────┘



### Query B — row count and date span
Claim: this is a real, bounded month-slice, not the whole 79M-row warehouse.

In [11]:
con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date,
        COUNT(DISTINCT report_date)     AS n_distinct_days
    FROM read_parquet('{MID_MONTH}')
""").show()


┌─────────┬───────────┬─────────────────┬────────────┬────────────┬─────────────────┐
│ n_rows  │ n_clients │ n_content_items │  min_date  │  max_date  │ n_distinct_days │
│  int64  │   int64   │      int64      │    date    │    date    │      int64      │
├─────────┼───────────┼─────────────────┼────────────┼────────────┼─────────────────┤
│ 9841378 │        55 │          331437 │ 2026-03-01 │ 2026-03-31 │              31 │
└─────────┴───────────┴─────────────────┴────────────┴────────────┴─────────────────┘



### Query C — availability, filtered with `IS TRUE`
Claim: not every row has usable data across both source systems. `skills/flyrank/flyrank-data/SKILL.md` documents `ga4_data_available` as a real boolean flag -- rows before a client's `ga4_data_start` are zero-filled for GA4 columns with this flag set `FALSE`. It also notes a third of clients have little or no usable search/analytics history. This lane's *features* are GSC-only, but proving I understand which rows are actually usable -- and by how much -- is the honest version of "availability" for this table.

In [12]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS n_ga4_available,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_ga4_available
    FROM read_parquet('{MID_MONTH}')
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────────────┬───────────────────┐
│ n_total │ n_ga4_available │ pct_ga4_available │
│  int64  │      int64      │      double       │
├─────────┼─────────────────┼───────────────────┤
│ 9841378 │          413966 │               4.2 │
└─────────┴─────────────────┴───────────────────┘



### Five features (max), each with an "available when?" line
All five are built **only** from days 1-15 of the month. Nothing from days 16+ touches a feature -- that half is reserved for the label.

In [13]:
raw = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{MID_MONTH}')
    WHERE gsc_data_available IS TRUE
""").df()

raw['period'] = raw['report_date'].apply(lambda d: 'first_half' if d.day <= 15 else 'second_half')
first_half = raw[raw['period'] == 'first_half']
second_half = raw[raw['period'] == 'second_half']

features = first_half.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions_first_half=('gsc_impressions', 'sum'),
    clicks_first_half=('gsc_clicks', 'sum'),
    avg_position_first_half=('gsc_avg_position', 'mean'),
    active_days_first_half=('report_date', 'nunique'),
).reset_index()

# CTR computed from summed clicks/impressions, not averaged daily ratios -- avoids letting
# low-impression days distort the rate.
features['ctr_first_half'] = features['clicks_first_half'] / features['impressions_first_half']

features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,active_days_first_half,ctr_first_half
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,1,12.639599,15,0.008403
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,6,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0,8.094074,15,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0,12.587226,15,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0,11.500000,5,0.000000


1. **`impressions_first_half`** -- knowable at the day-15 decision moment because it only sums rows with `report_date` in days 1-15, already observed by then.
2. **`clicks_first_half`** -- same reasoning: a sum over already-observed first-half rows.
3. **`ctr_first_half`** -- an average over the same first-half rows; second-half CTR is never touched.
4. **`avg_position_first_half`** -- same window restriction; today's rank isn't informed by anything that hasn't happened yet.
5. **`active_days_first_half`** -- a count of distinct observed days in the first half; can't reference the second half by construction.

### The trap -- deliberate label leakage
Label / proxy: `declined = 1` if `impressions_second_half < impressions_first_half`, else `0`. This uses the second half on purpose -- it's the outcome window, not a feature.

In [14]:
second_half_agg = second_half.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions_second_half=('gsc_impressions', 'sum'),
).reset_index()

labeled = features.merge(second_half_agg, on=['client_hash_id', 'content_hash_id'], how='inner')
labeled['declined'] = (labeled['impressions_second_half'] < labeled['impressions_first_half']).astype(int)

print(labeled['declined'].value_counts(normalize=True))


declined
0    0.603639
1    0.396361
Name: proportion, dtype: float64


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['impressions_first_half', 'clicks_first_half', 'ctr_first_half',
               'avg_position_first_half', 'active_days_first_half']

X, y = labeled[honest_cols], labeled['declined']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (first-half features only): {honest_auc:.3f}")

# --- THE TRAP: add ONE label-derived column on purpose ---
leaky_cols = honest_cols + ['impressions_second_half']  # this IS what the label is computed from

X_leak = labeled[leaky_cols]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, leaky_model.predict_proba(X_test_l)[:, 1])
print(f"Leaky AUC (with impressions_second_half included): {leaky_auc:.3f}")

# --- delete the leak, keep the honest number ---
print(f"\nKept: honest AUC = {honest_auc:.3f}. Deleted: impressions_second_half.")


Honest AUC (first-half features only): 0.640
Leaky AUC (with impressions_second_half included): 1.000

Kept: honest AUC = 0.640. Deleted: impressions_second_half.


**The leakage lesson (notebook 02, performed here on real warehouse data):** including `impressions_second_half` pushes AUC toward a near-perfect score, because it's arithmetically almost the same information the label was built from -- the model reads the answer key instead of learning a pattern. The only trustworthy number is the honest AUC from first-half-only features; that's the one that survives into the modeling weeks, and the leaky one gets deleted, not reported.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel:** `dim_clients.gsc_data_start` / `ga4_data_start` differ per client, and per `skills/flyrank/flyrank-data/SKILL.md` a third of clients have little or no usable search/analytics history at all -- `month=2026-03` does not guarantee the same client coverage as any other month.
- **Page-level only, no query-level detail:** `fact_content_query_90d` is deliberately excluded (Section 2), so nothing here speaks to which *queries* drove a page's impressions or clicks -- only the page-day aggregate.
- **Sealed test month:** `2026-06` (the `_sample`) is the natural outcome window for any past-to-future proxy built on earlier months, so no claim here is assumed to hold there -- it stays untouched until evaluation.
- **Directional proxy, not ground truth:** "declined" is a same-month split-half heuristic (no seasonality, algorithm-update, or site-change awareness) -- treat rankings from it as directional and decision-support, not measured fact.


In [16]:
# No additional query -- Section 4 is a written limitation, checked against the panel facts
# already surfaced in Query B/C above and the flyrank-data skill file, not a new number.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

> Run top-to-bottom in Colab yourself and check the two remaining boxes -- I could not execute against the gated HF dataset from this environment, so no cell above has been actually run.